# 05 — Selecao de Features Multi-Estagio

Pipeline de selecao de features em 5 estagios para o modelo de risco de credito (FPD).

| Estagio | Metodo | Criterio |
|---------|--------|----------|
| 1 | Information Value (IV) | IV > 0.02 |
| 2 | L1 Logistic Regression | coef != 0 |
| 3 | Correlacao | \|r\| > 0.90 drop |
| 4 | Estabilidade PSI | PSI > 0.25 drop |
| 5 | Anti-Leakage | proxy de target |

**Resultado**: 59 features selecionadas, salvas em artefato.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------------------
# Importar config e utils do projeto
# ---------------------------------------------------------------------------
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from config import *
from utils import *

print(f"Target: {TARGET}")
print(f"Train SAFRAs: {TRAIN_SAFRAS}")
print(f"OOT SAFRAs: {OOT_SAFRAS}")
print(f"Features selecionadas (referencia): {len(SELECTED_FEATURES)}")

In [ ]:
# ---------------------------------------------------------------------------
# Carregar feature store da Gold (parquet com projecao de colunas)
# ---------------------------------------------------------------------------
log("Carregando feature store da Gold...")

# Identificar colunas candidatas (excluir chaves e target)
key_cols = ["NUM_CPF", "SAFRA"]
meta_cols = key_cols + [TARGET]

df = pd.read_parquet(LOCAL_DATA_PATH, engine="pyarrow")
log(f"Shape: {df.shape}")
log(f"Taxa FPD global: {df[TARGET].mean():.4f} ({df[TARGET].sum():,} / {len(df):,})")

# Separar features candidatas
candidate_features = [c for c in df.columns if c not in meta_cols
                      and c not in ["_execution_id", "_data_inclusao",
                                    "_data_alteracao_silver", "DT_PROCESSAMENTO"]]
log(f"Features candidatas: {len(candidate_features)}")

# Split train/oot para selecao
df_train = df[df["SAFRA"].isin(TRAIN_SAFRAS)].copy()
df_oot = df[df["SAFRA"].isin(OOT_SAFRAS)].copy()
log(f"Train: {len(df_train):,} | OOT: {len(df_oot):,}")

## Estagio 1: Information Value (IV > 0.02)

O **Information Value (IV)** mede o poder preditivo de cada feature em relacao ao target.

| IV | Poder Preditivo |
|----|----------------|
| < 0.02 | Nao util |
| 0.02 - 0.10 | Fraco |
| 0.10 - 0.30 | Medio |
| 0.30 - 0.50 | Forte |
| > 0.50 | Suspeito (possivel leakage) |

Criterio: manter features com **IV > 0.02**.

In [ ]:
# ---------------------------------------------------------------------------
# Estagio 1: Information Value
# ---------------------------------------------------------------------------
log("Estagio 1: Calculando IV para todas as features...")


def compute_iv(df_in, feature, target, bins=10):
    """Calcula Information Value de uma feature numerica."""
    try:
        tmp = df_in[[feature, target]].dropna()
        if tmp[feature].nunique() < 2:
            return 0.0
        tmp["bin"] = pd.qcut(tmp[feature], bins, duplicates="drop")
        grouped = tmp.groupby("bin", observed=True)[target].agg(["sum", "count"])
        grouped["non_event"] = grouped["count"] - grouped["sum"]
        grouped["event_pct"] = grouped["sum"] / grouped["sum"].sum()
        grouped["non_event_pct"] = grouped["non_event"] / grouped["non_event"].sum()
        # Evitar log(0)
        grouped["event_pct"] = grouped["event_pct"].clip(lower=1e-6)
        grouped["non_event_pct"] = grouped["non_event_pct"].clip(lower=1e-6)
        grouped["woe"] = np.log(grouped["non_event_pct"] / grouped["event_pct"])
        grouped["iv"] = (grouped["non_event_pct"] - grouped["event_pct"]) * grouped["woe"]
        return grouped["iv"].sum()
    except Exception:
        return 0.0


iv_results = {}
for feat in candidate_features:
    iv_val = compute_iv(df_train, feat, TARGET)
    iv_results[feat] = round(iv_val, 6)

iv_df = pd.DataFrame([
    {"feature": k, "iv": v} for k, v in iv_results.items()
]).sort_values("iv", ascending=False).reset_index(drop=True)

# Filtro: IV > 0.02
IV_THRESHOLD = 0.02
features_iv = iv_df[iv_df["iv"] > IV_THRESHOLD]["feature"].tolist()

log(f"Features com IV > {IV_THRESHOLD}: {len(features_iv)} de {len(candidate_features)}")
print(f"\nTop 20 features por IV:")
print(iv_df.head(20).to_string(index=False))

## Estagio 2: L1 Logistic Regression (coef != 0)

A **Regressao Logistica com penalizacao L1 (Lasso)** promove esparsidade no vetor de
coeficientes, zerando automaticamente features irrelevantes.

Criterio: manter features com **coeficiente != 0** apos ajuste L1.

In [ ]:
# ---------------------------------------------------------------------------
# Estagio 2: L1 Logistic Regression
# ---------------------------------------------------------------------------
log("Estagio 2: Ajustando Logistic Regression L1...")

X_train_l1 = df_train[features_iv].copy()
y_train_l1 = df_train[TARGET].values

# Imputar e escalar
imputer = SimpleImputer(strategy="median")
X_train_l1_imp = imputer.fit_transform(X_train_l1)

scaler = StandardScaler()
X_train_l1_scaled = scaler.fit_transform(X_train_l1_imp)

lr_l1 = LogisticRegression(
    penalty="l1", C=1.0, solver="liblinear",
    class_weight="balanced", max_iter=1000, random_state=42
)
lr_l1.fit(X_train_l1_scaled, y_train_l1)

# Selecionar features com coeficiente != 0
coefs = pd.Series(lr_l1.coef_[0], index=features_iv)
features_l1 = coefs[coefs != 0].index.tolist()

log(f"Features com coef != 0: {len(features_l1)} de {len(features_iv)}")
log(f"Features eliminadas por L1: {len(features_iv) - len(features_l1)}")

## Estagio 3: Correlacao (|r| > 0.90 → drop)

Features altamente correlacionadas introduzem **multicolinearidade** e instabilidade nos
coeficientes. Para cada par com |r| > 0.90, removemos a feature que **nao** esta na
lista `SELECTED_FEATURES` (priorizando as ja validadas).

Criterio: **|r| > 0.90** entre qualquer par → drop a menos prioritaria.

In [ ]:
# ---------------------------------------------------------------------------
# Estagio 3: Correlation filter
# ---------------------------------------------------------------------------
log("Estagio 3: Calculando matriz de correlacao...")

CORR_THRESHOLD = 0.90

X_corr = df_train[features_l1].apply(pd.to_numeric, errors="coerce")
X_corr = X_corr.fillna(X_corr.median())
corr_matrix = X_corr.corr().abs()

# Identificar pares com alta correlacao
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = set()

for col_name in upper.columns:
    for row_name in upper.index:
        if upper.loc[row_name, col_name] > CORR_THRESHOLD:
            # Manter a que esta em SELECTED_FEATURES, dropar a outra
            if col_name in SELECTED_FEATURES and row_name not in SELECTED_FEATURES:
                to_drop_corr.add(row_name)
            elif row_name in SELECTED_FEATURES and col_name not in SELECTED_FEATURES:
                to_drop_corr.add(col_name)
            else:
                # Ambas ou nenhuma em SELECTED — drop a com menor IV
                iv_col = iv_results.get(col_name, 0)
                iv_row = iv_results.get(row_name, 0)
                to_drop_corr.add(col_name if iv_col < iv_row else row_name)

features_corr = [f for f in features_l1 if f not in to_drop_corr]

log(f"Pares com |r| > {CORR_THRESHOLD}: {len(to_drop_corr)} features removidas")
log(f"Features restantes: {len(features_corr)}")

## Estagio 4: Estabilidade PSI (PSI > 0.25 → drop)

O **Population Stability Index (PSI)** mede a estabilidade da distribuicao de uma feature
entre o periodo de treino e o periodo out-of-time (OOT).

| PSI | Interpretacao |
|-----|---------------|
| < 0.10 | Estavel (GREEN) |
| 0.10 - 0.25 | Alerta (YELLOW) |
| > 0.25 | Instavel (RED) — drop |

Criterio: **PSI > 0.25** → remover feature.

In [ ]:
# ---------------------------------------------------------------------------
# Estagio 4: PSI stability filter
# ---------------------------------------------------------------------------
log("Estagio 4: Calculando PSI (train vs OOT) por feature...")

PSI_THRESHOLD = 0.25

psi_results = {}
for feat in features_corr:
    try:
        train_vals = df_train[feat].dropna().values.astype(float)
        oot_vals = df_oot[feat].dropna().values.astype(float)
        if len(train_vals) > 0 and len(oot_vals) > 0:
            psi_val = compute_psi(train_vals, oot_vals)
        else:
            psi_val = 0.0
    except Exception:
        psi_val = 0.0
    psi_results[feat] = psi_val

psi_df = pd.DataFrame([
    {"feature": k, "psi": v, "alert": psi_alert(v)} for k, v in psi_results.items()
]).sort_values("psi", ascending=False).reset_index(drop=True)

to_drop_psi = psi_df[psi_df["psi"] > PSI_THRESHOLD]["feature"].tolist()
features_psi = [f for f in features_corr if f not in to_drop_psi]

log(f"Features com PSI > {PSI_THRESHOLD}: {len(to_drop_psi)} removidas")
if to_drop_psi:
    print(f"Features instaveis removidas: {to_drop_psi}")
log(f"Features restantes: {len(features_psi)}")

print(f"\nDistribuicao de alertas PSI:")
print(psi_df["alert"].value_counts().to_string())

## Estagio 5: Validacao Anti-Leakage

Verificacao final para garantir que nenhuma feature restante seja um **proxy direto do target**.

Colunas bloqueadas:
- `FAT_VLR_FPD` (valor direto de FPD na fatura)
- Qualquer coluna contendo `VLR_FPD` no nome
- Qualquer coluna com correlacao > 0.95 com o target

In [ ]:
# ---------------------------------------------------------------------------
# Estagio 5: Anti-leakage validation
# ---------------------------------------------------------------------------
log("Estagio 5: Validacao anti-leakage...")

LEAKAGE_PATTERNS = ["VLR_FPD", "TARGET_FPD", "_LEAKAGE"]

leakage_found = []
for feat in features_psi:
    # Pattern check
    if any(pattern in feat.upper() for pattern in LEAKAGE_PATTERNS):
        leakage_found.append((feat, "pattern_match"))
        continue
    # High correlation with target check
    try:
        corr_with_target = abs(df_train[feat].corr(df_train[TARGET].astype(float)))
        if corr_with_target > 0.95:
            leakage_found.append((feat, f"corr_target={corr_with_target:.4f}"))
    except Exception:
        pass

if leakage_found:
    log(f"ALERTA: {len(leakage_found)} features com possivel leakage detectadas!")
    for feat, reason in leakage_found:
        print(f"  REMOVIDA: {feat} ({reason})")
    leakage_names = [f[0] for f in leakage_found]
    features_final = [f for f in features_psi if f not in leakage_names]
else:
    features_final = features_psi.copy()
    log("Nenhuma feature com leakage detectada. Pipeline limpo.")

log(f"Features finais apos anti-leakage: {len(features_final)}")

## Resultado Final: 59 Features Selecionadas

Resumo do funil de selecao multi-estagio e lista final de features aprovadas.

In [ ]:
# ---------------------------------------------------------------------------
# Resultado final: lista de features e ranking
# ---------------------------------------------------------------------------
print("=" * 70)
print("FUNIL DE SELECAO DE FEATURES")
print("=" * 70)
print(f"Candidatas iniciais:        {len(candidate_features)}")
print(f"Apos IV > 0.02:             {len(features_iv)}")
print(f"Apos L1 (coef != 0):        {len(features_l1)}")
print(f"Apos correlacao (|r|<0.90): {len(features_corr)}")
print(f"Apos PSI < 0.25:            {len(features_psi)}")
print(f"Apos anti-leakage:          {len(features_final)}")
print("=" * 70)

# Ranking table com IV e PSI
ranking = pd.DataFrame({"feature": features_final})
ranking["iv"] = ranking["feature"].map(iv_results)
ranking["psi"] = ranking["feature"].map(psi_results)
ranking["psi_alert"] = ranking["psi"].apply(psi_alert)
ranking = ranking.sort_values("iv", ascending=False).reset_index(drop=True)
ranking.index += 1
ranking.index.name = "rank"

print(f"\nRanking das {len(features_final)} features selecionadas:")
print(ranking.to_string())

# Salvar artefato
os.makedirs(ARTIFACT_DIR, exist_ok=True)
artifact_path = os.path.join(ARTIFACT_DIR, "selected_features.json")
import json
with open(artifact_path, "w") as f:
    json.dump({
        "n_features": len(features_final),
        "features": features_final,
        "ranking": ranking[["feature", "iv", "psi", "psi_alert"]].to_dict(orient="records"),
        "funnel": {
            "candidates": len(candidate_features),
            "after_iv": len(features_iv),
            "after_l1": len(features_l1),
            "after_corr": len(features_corr),
            "after_psi": len(features_psi),
            "after_leakage": len(features_final),
        }
    }, f, indent=2)
log(f"Artefato salvo em {artifact_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Plots: IV bar chart, correlation heatmap, PSI bar chart
# ---------------------------------------------------------------------------
plots_dir = os.path.join(ARTIFACT_DIR, "plots")
os.makedirs(plots_dir, exist_ok=True)

# --- Plot 1: IV bar chart (top 30) ---
fig, ax = plt.subplots(figsize=(12, 8))
top30_iv = iv_df[iv_df["feature"].isin(features_final)].head(30)
colors = ["#2ecc71" if v > 0.10 else "#f39c12" if v > 0.05 else "#3498db"
          for v in top30_iv["iv"]]
ax.barh(top30_iv["feature"][::-1], top30_iv["iv"][::-1], color=colors[::-1])
ax.set_xlabel("Information Value (IV)")
ax.set_title("Top 30 Features por Information Value")
ax.axvline(x=0.02, color="red", linestyle="--", alpha=0.7, label="Threshold (0.02)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "iv_top30.png"), dpi=150)
plt.show()
log("Plot IV salvo.")

# --- Plot 2: Correlation heatmap (features selecionadas) ---
fig, ax = plt.subplots(figsize=(16, 14))
corr_selected = df_train[features_final].apply(pd.to_numeric, errors="coerce").corr()
mask = np.triu(np.ones_like(corr_selected, dtype=bool))
sns.heatmap(corr_selected, mask=mask, cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, ax=ax, square=True,
            xticklabels=True, yticklabels=True,
            cbar_kws={"shrink": 0.8, "label": "Correlacao"})
ax.set_title("Matriz de Correlacao — Features Selecionadas")
ax.tick_params(axis="both", labelsize=6)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "correlation_heatmap.png"), dpi=150)
plt.show()
log("Plot correlacao salvo.")

# --- Plot 3: PSI per feature bar chart ---
fig, ax = plt.subplots(figsize=(12, 8))
psi_selected = psi_df[psi_df["feature"].isin(features_final)].sort_values("psi", ascending=False).head(30)
colors_psi = ["#e74c3c" if a == "RED" else "#f39c12" if a == "YELLOW" else "#2ecc71"
              for a in psi_selected["alert"]]
ax.barh(psi_selected["feature"][::-1], psi_selected["psi"][::-1], color=colors_psi[::-1])
ax.set_xlabel("PSI (Train vs OOT)")
ax.set_title("PSI por Feature — Estabilidade Temporal")
ax.axvline(x=0.10, color="#f39c12", linestyle="--", alpha=0.7, label="Alerta (0.10)")
ax.axvline(x=0.25, color="#e74c3c", linestyle="--", alpha=0.7, label="Critico (0.25)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "psi_features.png"), dpi=150)
plt.show()
log("Plot PSI salvo.")

print(f"\nTodos os plots salvos em {plots_dir}")

## Gold Final: Persistir Features Selecionadas

Escreve o dataset final `clientes_final` no Gold com apenas as 59 features selecionadas + chaves + target.

| Coluna | Descricao |
|--------|----------|
| `NUM_CPF` | Chave primaria |
| `SAFRA` | Safra (particionamento) |
| `FPD` | Target |
| 59 features | Features selecionadas pelo pipeline multi-estagio |

In [ ]:
# Inicializar SparkSession para escrita Gold
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count as spark_count

builder = SparkSession.builder.appName("gold-final-write")
for key, value in SPARK_CONFIG.items():
    builder = builder.config(key, value)
spark = builder.getOrCreate()

log(f"SparkSession criada: {spark.version}")

# Ler clientes_consolidado do Gold
log("Lendo clientes_consolidado do Gold...")
df_consolidated = spark.read.format("delta").load(GOLD_CONSOLIDATED_PATH)
log(f"Consolidado: {df_consolidated.count():,} linhas, {len(df_consolidated.columns)} colunas")

# Projetar para colunas selecionadas + keys + target
final_cols = ["NUM_CPF", "SAFRA", TARGET] + features_final
# Verificar que todas as colunas existem
missing_cols = [c for c in final_cols if c not in df_consolidated.columns]
if missing_cols:
    log(f"AVISO: Colunas ausentes no consolidado: {missing_cols}")
    final_cols = [c for c in final_cols if c in df_consolidated.columns]

df_final = df_consolidated.select(*final_cols)
log(f"Gold Final: {len(df_final.columns)} colunas projetadas")

# Escrever como Delta particionado por SAFRA
target_uri = GOLD_FINAL_PATH
df_final.write.mode("overwrite").partitionBy("SAFRA") \
    .format("delta").option("overwriteSchema", "true").save(target_uri)

final_rows = df_final.count()
final_cols_count = len(df_final.columns)
log(f"[GOLD] clientes_final escrito: {final_rows:,} linhas, {final_cols_count} colunas")
log(f"[GOLD] Target: {target_uri}")

In [ ]:
# Validacao do Gold Final
print(f"{'='*60}")
print(f"VALIDACAO — GOLD FINAL")
print(f"{'='*60}")
checks_pass = 0
checks_fail = 0

# Check 1: Row count == consolidated
consolidated_rows = df_consolidated.count()
if final_rows == consolidated_rows:
    print(f"  [PASS] Row count: {final_rows:,} == consolidated ({consolidated_rows:,})")
    checks_pass += 1
else:
    print(f"  [FAIL] Row count: {final_rows:,} != consolidated ({consolidated_rows:,})")
    checks_fail += 1

# Check 2: Col count == features + 3 keys (NUM_CPF, SAFRA, FPD)
expected_cols = len(features_final) + 3
if final_cols_count == expected_cols:
    print(f"  [PASS] Col count: {final_cols_count} == {len(features_final)} features + 3 keys")
    checks_pass += 1
else:
    print(f"  [FAIL] Col count: {final_cols_count} (esperado: {expected_cols})")
    checks_fail += 1

# Check 3: PK uniqueness
df_final_read = spark.read.format("delta").load(target_uri)
dup_count = df_final_read.groupBy("NUM_CPF", "SAFRA").agg(spark_count("*").alias("cnt")).filter(col("cnt") > 1).count()
if dup_count == 0:
    print(f"  [PASS] PK uniqueness OK")
    checks_pass += 1
else:
    print(f"  [FAIL] {dup_count:,} PK duplicates")
    checks_fail += 1

print(f"\n  Resultado: {checks_pass} PASS / {checks_fail} FAIL")
print(f"{'='*60}")